In [ ]:
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import ToTensor

training_data = datasets.FashionMNIST(
  root="data",
  train=True,
  download=True,
  transform=ToTensor()
)

test_data = datasets.FashionMNIST(
  root="data",
  train=False,
  download=True,
  transform=ToTensor()
)

train_dataloader = DataLoader(training_data, batch_size=64)
test_dataloader = DataLoader(test_data, batch_size=64)

class NeuralNetwork(nn.Module):
  def __init__(self):
    super.__init__()
    self.flatten = nn.Flatten()
    self.linear_relu_stack = nn.Sequential(
      nn.Linear(28 * 28, 512),
      nn.ReLU(),
      nn.Linear(512, 512),
      nn.ReLU(),
      nn.Linear(512, 10),
    )

  def forward(self, x):
    x = self.flatten(x)
    logits = self.linear_relu_stack(x)
    return logits
  
model = NeuralNetwork()

100%|██████████| 26.4M/26.4M [00:49<00:00, 534kB/s]
100%|██████████| 29.5k/29.5k [00:00<00:00, 98.6kB/s]
100%|██████████| 4.42M/4.42M [00:08<00:00, 514kB/s]
100%|██████████| 5.15k/5.15k [00:00<00:00, 1.61MB/s]

Dataset FashionMNIST
    Number of datapoints: 60000
    Root location: data
    Split: Train
    StandardTransform
Transform: ToTensor() Dataset FashionMNIST
    Number of datapoints: 10000
    Root location: data
    Split: Test
    StandardTransform
Transform: ToTensor()


In [ ]:
# 전체 dataset을 몇 번 반복할지 결정하는 횟수
epoch = 5
# dataset을 몇 개로 묶을지 결정하는 개수
batch_size = 64
# 가중치를 얼마나 업데이트하는가?
# learning_rate * gradient
# 너무 크면 가중치가 널뛰어서 학습이 불안정하고 너무 작으면 학습이 느리다.
learning_rate = 1e-3

In [ ]:
# 손실 함수
loss_fn = nn.CrossEntropyLoss()

# 최적화
optimizer = torch.optim.SGD(model.parameters(), lr=learning_rate)

In [ ]:
def train_loop(dataloader, model, loss_fn, optimizer):
  size = len(dataloader.dataset)

  model.train()
  for batch, (X, y) in enumerate(dataloader):
    pred = model(X)
    loss = loss_fn(pred, y)

    loss.backward
    # step() : 최적화로 손실 함수 기반으로 계산된 기울기 방향으로 가중치를 업데이트 한다.
    optimizer.step()
    # 가중치가 누적되면 엉뚱한 각 특징을 엉뚱한 방향으로 학습하게 되기 때문에 batch마다 초기화한다.
    optimizer.zero_grad()

    if batch % 100 == 0:
      loss, current = loss.item(), batch * batch_size + len(X)
      print(f"loss: {loss:>7f}  [{current:>5d}/{size:>5d}]")

def test_loop(dataloader, model, loss_fn):
  model.eval()
  size = len(dataloader.dataset)
  num_batches = len(dataloader)
  test_loss, correct = 0, 0

  with torch.no_grad():
    for X, y in dataloader:
      pred = model(X)
      test_loss += loss_fn(pred, y).item()
      correct += (pred.argmax(1) == y).type(torch.float).sum().item()

  test_loss /= num_batches
  correct /= size 
  print(f"Test Error: \n Accuracy: {(100*correct):>0.1f}%, Avg loss: {test_loss:>8f} \n")

In [ ]:
for t in range(epoch):
  print(f"Epoch {t+1}\n-------------------------------")
  train_loop(train_dataloader, model, loss_fn, optimizer)
  test_loop(test_dataloader, model, loss_fn)
print("Done!")